# Notebook 6: SQL (06_SQL.ipynb)

**Sprint:** Sprint 1 – AI/ML Engineering Foundations

**Note:** This notebook uses SQLite (via Python's built-in sqlite3 
module) to demonstrate SQL concepts - no separate database server 
required. The SQL syntax covered here applies directly to MySQL as well.

In [3]:
import sqlite3

conn = sqlite3.connect(":memory:")
cursor = conn.cursor()

print("Database connection ready!")

Database connection ready!


## SQL: 1. Database and Tables

### Explanation
A Database is an organized collection of data. A Table is where 
that data lives, organized into rows (records) and columns 
(fields) - similar to a spreadsheet, but with strict structure.

In [4]:
cursor.execute("""
CREATE TABLE students (
    id INTEGER,
    name TEXT,
    course TEXT,
    marks INTEGER
)
""")

cursor.execute("INSERT INTO students VALUES (1, 'Balaji', 'AI/ML', 95)")
cursor.execute("INSERT INTO students VALUES (2, 'Priya', 'Web Dev', 88)")
conn.commit()

cursor.execute("SELECT * FROM students")
print(cursor.fetchall())

# Where it's used: CREATE TABLE students (...) defines the table
# structure. Each INSERT INTO adds one row of data. conn.commit()
# saves the changes.

[(1, 'Balaji', 'AI/ML', 95), (2, 'Priya', 'Web Dev', 88)]


## SQL: 2 & 3. Primary Key and Foreign Key

### Explanation
A Primary Key uniquely identifies each row in a table - no two 
rows can share the same value. A Foreign Key is a column that 
refers to another table's Primary Key, linking the two tables together.

In [5]:
cursor.execute("""
CREATE TABLE courses (
    course_id INTEGER PRIMARY KEY,
    course_name TEXT
)
""")

cursor.execute("""
CREATE TABLE enrollments (
    student_id INTEGER,
    course_id INTEGER,
    FOREIGN KEY (course_id) REFERENCES courses(course_id)
)
""")

cursor.execute("INSERT INTO courses VALUES (1, 'AI/ML')")
cursor.execute("INSERT INTO enrollments VALUES (1, 1)")
conn.commit()
print("Tables created successfully!")

# Where it's used: course_id INTEGER PRIMARY KEY makes each course_id
# unique. FOREIGN KEY (course_id) REFERENCES courses(course_id) links
# enrollments back to courses.

Tables created successfully!


## SQL: 4 & 5. Relationships and Constraints

### Explanation
A Relationship describes how tables connect to each other (e.g., 
one student can enroll in many courses - a "one-to-many" 
relationship). Constraints are rules enforced on columns - like 
NOT NULL (can't be empty) or UNIQUE (no duplicates) - to keep data valid.

In [7]:
cursor.execute("""
CREATE TABLE teachers (
    teacher_id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    email TEXT UNIQUE
)
""")

cursor.execute("INSERT INTO teachers VALUES (1, 'Mr. Kumar', 'kumar@school.com')")
conn.commit()

try:
    cursor.execute("INSERT INTO teachers VALUES (2, 'Mr. Raj', 'kumar@school.com')")  # duplicate email
    conn.commit()
except sqlite3.IntegrityError as e:
    print("Constraint violation:", e)

# Where it's used: NOT NULL forces 'name' to always have a value.
# UNIQUE on 'email' means no two teachers can share the same email -
# the second insert fails because 'kumar@school.com' is duplicated,
# demonstrating the constraint in action.

Constraint violation: UNIQUE constraint failed: teachers.email


## SQL: 6. Adding more sample data (for meaningful queries ahead)

In [8]:
cursor.execute("INSERT INTO students VALUES (3, 'Arun', 'AI/ML', 78)")
cursor.execute("INSERT INTO students VALUES (4, 'Divya', 'Web Dev', 91)")
cursor.execute("INSERT INTO students VALUES (5, 'Karthik', 'AI/ML', 65)")
conn.commit()

cursor.execute("SELECT * FROM students")
print(cursor.fetchall())

[(1, 'Balaji', 'AI/ML', 95), (2, 'Priya', 'Web Dev', 88), (3, 'Arun', 'AI/ML', 78), (4, 'Divya', 'Web Dev', 91), (5, 'Karthik', 'AI/ML', 65)]


## SQL: 7 & 8. SELECT and WHERE

### Explanation
SELECT retrieves specific columns (or all, using *) from a table. 
WHERE filters rows based on a condition, so only matching rows are 
returned.

In [9]:
cursor.execute("SELECT name, marks FROM students")
print("All names and marks:", cursor.fetchall())

cursor.execute("SELECT * FROM students WHERE marks > 80")
print("Students with marks > 80:", cursor.fetchall())

# Where it's used: 'SELECT name, marks FROM students' picks only
# those two columns. 'WHERE marks > 80' filters rows, returning
# only students whose marks exceed 80 - out of 5 students, only
# the matching ones come back.

All names and marks: [('Balaji', 95), ('Priya', 88), ('Arun', 78), ('Divya', 91), ('Karthik', 65)]
Students with marks > 80: [(1, 'Balaji', 'AI/ML', 95), (2, 'Priya', 'Web Dev', 88), (4, 'Divya', 'Web Dev', 91)]


## SQL: 9. ORDER BY

### Explanation
ORDER BY sorts the result rows by a specified column - ascending 
(ASC, default) or descending (DESC).

In [10]:
cursor.execute("SELECT name, marks FROM students ORDER BY marks DESC")
print("Sorted by marks (highest first):", cursor.fetchall())

# Where it's used: 'ORDER BY marks DESC' sorts the results so the
# highest marks appear first - without this, rows would come back
# in whatever order they were inserted.

Sorted by marks (highest first): [('Balaji', 95), ('Divya', 91), ('Priya', 88), ('Arun', 78), ('Karthik', 65)]


## SQL: 10 & 11. GROUP BY and HAVING

### Explanation
GROUP BY groups rows that share the same value in a column, 
usually combined with an aggregate function (like COUNT, AVG). 
HAVING filters those GROUPED results - similar to WHERE, but WHERE 
filters individual rows, while HAVING filters groups.

In [11]:
cursor.execute("SELECT course, COUNT(*) FROM students GROUP BY course")
print("Students per course:", cursor.fetchall())

cursor.execute("SELECT course, AVG(marks) FROM students GROUP BY course HAVING AVG(marks) > 75")
print("Courses with avg marks > 75:", cursor.fetchall())

# Where it's used: 'GROUP BY course' bundles all students into
# groups by their course, and COUNT(*) counts how many are in each
# group. 'HAVING AVG(marks) > 75' then filters OUT any group whose
# average doesn't meet that condition - HAVING works on the
# grouped/aggregated result, not individual rows.

Students per course: [('AI/ML', 3), ('Web Dev', 2)]
Courses with avg marks > 75: [('AI/ML', 79.33333333333333), ('Web Dev', 89.5)]


## SQL: 12 & 13. DISTINCT and LIMIT

### Explanation
DISTINCT removes duplicate values from the result, showing each 
unique value only once. LIMIT restricts how many rows are 
returned, useful for previewing large datasets.

In [12]:
cursor.execute("SELECT DISTINCT course FROM students")
print("Unique courses:", cursor.fetchall())

cursor.execute("SELECT * FROM students ORDER BY marks DESC LIMIT 2")
print("Top 2 students by marks:", cursor.fetchall())

# Where it's used: 'DISTINCT course' removes duplicate course names -
# even though 3 students take AI/ML, it appears only once. 
# 'LIMIT 2' cuts the result down to just the top 2 rows after sorting.

Unique courses: [('AI/ML',), ('Web Dev',)]
Top 2 students by marks: [(1, 'Balaji', 'AI/ML', 95), (4, 'Divya', 'Web Dev', 91)]


## SQL: 14. Setup - Table with unmatched rows (for JOIN comparisons)

### Explanation
To see the real difference between JOIN types, we need rows in one 
table that DON'T match anything in the other - some students not 
enrolled anywhere, and one enrollment pointing to a non-existent course.

In [13]:
cursor.execute("INSERT INTO courses VALUES (2, 'Data Science')")   # no student enrolled in this
cursor.execute("INSERT INTO enrollments VALUES (2, NULL)")          # student with no course
conn.commit()

cursor.execute("SELECT * FROM courses")
print("Courses:", cursor.fetchall())
cursor.execute("SELECT * FROM enrollments")
print("Enrollments:", cursor.fetchall())

Courses: [(1, 'AI/ML'), (2, 'Data Science')]
Enrollments: [(1, 1), (2, None)]


## SQL: 15. INNER JOIN

### Explanation
INNER JOIN returns only the rows that have MATCHING values in 
BOTH tables. Non-matching rows from either side are excluded entirely.

In [14]:
cursor.execute("""
SELECT students.name, courses.course_name
FROM students
INNER JOIN enrollments ON students.id = enrollments.student_id
INNER JOIN courses ON enrollments.course_id = courses.course_id
""")
print("INNER JOIN result:", cursor.fetchall())

# Where it's used: only rows where student_id matches in enrollments
# AND course_id matches in courses come back. Students with no
# enrollment, or enrollments pointing to a missing course, are excluded.

INNER JOIN result: [('Balaji', 'AI/ML')]


## SQL: 16. LEFT JOIN

### Explanation
LEFT JOIN returns ALL rows from the LEFT (first) table, plus 
matching rows from the right table. If there's no match, the right 
side's columns show as NULL instead of dropping the row entirely.

In [15]:
cursor.execute("""
SELECT students.name, courses.course_name
FROM students
LEFT JOIN enrollments ON students.id = enrollments.student_id
LEFT JOIN courses ON enrollments.course_id = courses.course_id
""")
print("LEFT JOIN result:", cursor.fetchall())

# Where it's used: EVERY student appears here, even ones with no
# enrollment - their course_name just shows as None (NULL), instead
# of being excluded like in INNER JOIN.

LEFT JOIN result: [('Balaji', 'AI/ML'), ('Priya', None), ('Arun', None), ('Divya', None), ('Karthik', None)]


## SQL: 17, 18, 19. RIGHT JOIN, FULL JOIN, CROSS JOIN

### Explanation
RIGHT JOIN is the mirror of LEFT JOIN - all rows from the RIGHT 
table, plus matches from the left. FULL JOIN returns all rows from 
BOTH tables, matched where possible, NULL where not. CROSS JOIN 
combines every row from one table with every row from the other 
(no matching condition at all) - a full combination of both.

Note: RIGHT JOIN and FULL JOIN require a fairly recent SQLite 
version (3.39+). If you get an error, it just means your Python's 
built-in SQLite is older - the concept explanation still applies, 
just mention that in your notes.

In [16]:
try:
    cursor.execute("""
    SELECT students.name, courses.course_name
    FROM students
    RIGHT JOIN enrollments ON students.id = enrollments.student_id
    RIGHT JOIN courses ON enrollments.course_id = courses.course_id
    """)
    print("RIGHT JOIN result:", cursor.fetchall())
except sqlite3.OperationalError as e:
    print("RIGHT JOIN not supported in this SQLite version:", e)

try:
    cursor.execute("""
    SELECT students.name, courses.course_name
    FROM students
    FULL JOIN enrollments ON students.id = enrollments.student_id
    """)
    print("FULL JOIN result:", cursor.fetchall())
except sqlite3.OperationalError as e:
    print("FULL JOIN not supported in this SQLite version:", e)

# CROSS JOIN - always supported, combines EVERY row with EVERY row
cursor.execute("SELECT students.name, courses.course_name FROM students CROSS JOIN courses")
print("CROSS JOIN result (first few):", cursor.fetchall()[:6])

# Where it's used: CROSS JOIN has no ON condition at all - it pairs
# every single student with every single course, producing
# (number of students) x (number of courses) total rows.

RIGHT JOIN result: [('Balaji', 'AI/ML'), (None, 'Data Science')]
FULL JOIN not supported in this SQLite version: no such column: courses.course_name
CROSS JOIN result (first few): [('Balaji', 'AI/ML'), ('Balaji', 'Data Science'), ('Priya', 'AI/ML'), ('Priya', 'Data Science'), ('Arun', 'AI/ML'), ('Arun', 'Data Science')]


## SQL: 20. UNION

### Explanation
UNION combines the results of two SELECT queries into one result 
set, removing duplicates automatically. Both queries must return 
the same number of columns.

In [17]:
cursor.execute("""
SELECT name FROM students WHERE course = 'AI/ML'
UNION
SELECT name FROM students WHERE marks > 85
""")
print("UNION result:", cursor.fetchall())

# Where it's used: the first query gets AI/ML students, the second
# gets students with marks > 85. UNION merges both lists into one,
# automatically removing any name that appears in both results.

UNION result: [('Arun',), ('Balaji',), ('Divya',), ('Karthik',), ('Priya',)]


## SQL: 21. CASE

### Explanation
CASE works like an if/elif/else inside SQL - it evaluates 
conditions in order and returns a different value depending on 
which one matches, useful for creating custom labels/categories in results.

In [18]:
cursor.execute("""
SELECT name, marks,
    CASE
        WHEN marks >= 90 THEN 'Excellent'
        WHEN marks >= 75 THEN 'Good'
        ELSE 'Needs Improvement'
    END AS grade
FROM students
""")
print("CASE result:", cursor.fetchall())

# Where it's used: CASE checks each student's marks against the
# WHEN conditions in order, and 'grade' becomes whichever label
# matches first - same logic as if/elif/else in Python.

CASE result: [('Balaji', 95, 'Excellent'), ('Priya', 88, 'Good'), ('Arun', 78, 'Good'), ('Divya', 91, 'Excellent'), ('Karthik', 65, 'Needs Improvement')]


## SQL: 22. Subqueries

### Explanation
A subquery is a SELECT query nested INSIDE another query - used 
when you need the result of one query as input to another, rather 
than a fixed value.

In [19]:
cursor.execute("""
SELECT name, marks FROM students
WHERE marks > (SELECT AVG(marks) FROM students)
""")
print("Students above average:", cursor.fetchall())

# Where it's used: '(SELECT AVG(marks) FROM students)' is the
# subquery - it runs FIRST, calculating the average marks. The
# outer query then uses that calculated value to filter students
# scoring above it, without us hardcoding the average manually.

Students above average: [('Balaji', 95), ('Priya', 88), ('Divya', 91)]


## SQL: 23. Common Table Expressions (CTEs)

### Explanation
A CTE is a temporary named result set, defined using WITH, that 
you can reference like a table within the same query. It makes 
complex queries more readable by breaking them into named steps, 
instead of nesting subqueries deeply.

In [20]:
cursor.execute("""
WITH high_scorers AS (
    SELECT name, marks FROM students WHERE marks > 80
)
SELECT * FROM high_scorers ORDER BY marks DESC
""")
print("CTE result:", cursor.fetchall())

# Where it's used: 'WITH high_scorers AS (...)' defines a temporary
# named result (high_scorers), which the final SELECT then queries
# just like a regular table - making the logic easier to read than
# nesting the same subquery directly inside the outer query.

CTE result: [('Balaji', 95), ('Divya', 91), ('Priya', 88)]


## SQL: 24. Views

### Explanation
A View is a saved, reusable query stored under a name - you can 
SELECT from it just like a table, without rewriting the underlying 
query every time. It doesn't store data itself; it re-runs the 
original query each time you use it.

In [21]:
cursor.execute("""
CREATE VIEW ai_ml_students AS
SELECT name, marks FROM students WHERE course = 'AI/ML'
""")
conn.commit()

cursor.execute("SELECT * FROM ai_ml_students")
print("View result:", cursor.fetchall())

# Where it's used: 'CREATE VIEW ai_ml_students AS (...)' saves that
# query under a reusable name. From now on, 'SELECT * FROM
# ai_ml_students' works exactly like querying a real table, even
# though it's really just re-running the saved SELECT each time.

View result: [('Balaji', 95), ('Arun', 78), ('Karthik', 65)]


## SQL: 25. Indexes

### Explanation
An Index speeds up searching/filtering on a column, similar to a 
book's index letting you find a topic without reading every page. 
It adds a small storage/write overhead, but makes SELECT queries 
on that column much faster, especially on large tables.

In [22]:
cursor.execute("CREATE INDEX idx_marks ON students(marks)")
conn.commit()

cursor.execute("SELECT * FROM students WHERE marks > 80")
print("Query using indexed column:", cursor.fetchall())

# Where it's used: 'CREATE INDEX idx_marks ON students(marks)'
# builds an index on the marks column. The SELECT query filtering
# by marks can now use that index internally to find matching rows
# faster - the query result looks the same, but on a large table
# this would be noticeably quicker than scanning every row.

Query using indexed column: [(2, 'Priya', 'Web Dev', 88), (4, 'Divya', 'Web Dev', 91), (1, 'Balaji', 'AI/ML', 95)]
